In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Life Expectancy Forecasting

This notebook is a lightweight wrapper around the production script in `src/ml_forecasting.py`.
All forecasting logic lives in the script so there is only one source of truth.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_forecasting import run_forecasting_pipeline

run_forecasting_pipeline()


## Forecast results visualizations

Once the pipeline runs, this section loads the saved CatBoost outputs and shows the key charts again.

In [75]:
import pandas as pd
import plotly.express as px

forecast_file = PROJECT_ROOT / 'data' / 'results' / 'forecast_catboost.csv'
if forecast_file.exists():
    forecast_df = pd.read_csv(forecast_file)
    forecast_df = forecast_df.rename(columns={
        'Country Name': 'country_name',
        'Country Code': 'country_code',
        'Year': 'year',
        'Predicted': 'predicted_life_expectancy',
        'Actual': 'actual_life_expectancy',
    })

    # summary metrics across all forecast rows
    forecast_df['abs_error'] = (forecast_df['actual_life_expectancy'] - forecast_df['predicted_life_expectancy']).abs()
    summary = {
        'MAE': round(forecast_df['abs_error'].mean(), 4),
        'RMSE': round(((forecast_df['actual_life_expectancy'] - forecast_df['predicted_life_expectancy']) ** 2).mean() ** 0.5, 4),
        'Countries': int(forecast_df['country_name'].nunique()),
        'Years': int(forecast_df['year'].nunique()),
    }

    print('Forecast summary:')
    for k, v in summary.items():
        print(f'  {k}: {v}')

    country_options = sorted(forecast_df['country_name'].dropna().unique())
    selected_country = country_options[0] if country_options else None

    if selected_country is not None:
        country_df = forecast_df[forecast_df['country_name'] == selected_country].sort_values('year')
        country_plot = country_df[['year', 'actual_life_expectancy', 'predicted_life_expectancy']].melt(
            id_vars='year',
            var_name='series',
            value_name='life_expectancy',
        )
        fig = px.line(
            country_plot,
            x='year',
            y='life_expectancy',
            color='series',
            title=f'CatBoost Forecast vs Actual for {selected_country}',
            labels={'year': 'Year', 'life_expectancy': 'Life Expectancy (Years)', 'series': 'Series'},
            template='plotly_white',
        )
        fig.show()

    # country comparison: average absolute error by country
    country_error = (
        forecast_df.assign(error_abs=lambda d: (d['actual_life_expectancy'] - d['predicted_life_expectancy']).abs())
        .groupby('country_name', as_index=False)['error_abs']
        .mean()
        .sort_values('error_abs', ascending=False)
        .head(10)
    )

    comparison_fig = px.bar(
        country_error,
        x='country_name',
        y='error_abs',
        color='error_abs',
        title='Top 10 Countries by Average Absolute Forecast Error',
        labels={'country_name': 'Country', 'error_abs': 'Mean Absolute Error (Years)'},
        template='plotly_white',
    )
    comparison_fig.update_xaxes(tickangle=45)
    comparison_fig.show()
else:
    print('Forecast file not found yet. Run the forecasting pipeline first.')

Forecast summary:
  MAE: 0.9615
  RMSE: 1.7904
  Countries: 264
  Years: 13
